# Audit of the Phase 1 baseline (Antoine's sphere-rule notebook)

**Purpose.** Phase 1 ran Antoine's method (`reference/antoine_sphere_rule_edition2.ipynb`) with five KG embedding models. This notebook checks, one by one, the assumptions the method relies on. For each point it shows:

1. **Claim**: what goes wrong.
2. **Antoine's code**: the lines involved, copied from the reference notebook.
3. **Demonstration**: a small, self-contained experiment, and, where available, the evidence from the Phase 1 runs.
4. **Consequence** and a **possible fix**.

The aim is not to criticise the original work: the method was a reasonable first design. The aim is to show which parts of it need to change before its results can be interpreted, and why.

**How to run.** From the repository root, `cd phase2/notebooks && jupyter notebook part1a_audit_antoine.ipynb` (on the server it is executed automatically by `run_phase2.sh`; the evidence from the runs is in `part1b_evidence_antoine.ipynb`). Pure demonstrations run anywhere (they use random vectors as stand-in embeddings and take seconds). Cells marked *Phase 1 evidence* read the finished runs in `phase1/runs/` and the MRR results in `phase1/quality/`; they print a note and skip if those are not there yet. Set `RUN_HEAVY = True` to also load the full 1.7M-triple stream (needs ~10 GB RAM, a few minutes).

| # | Problem | Severity |
|---|---|---|
| 1 | The sphere retrain rule cannot fire | critical |
| 2 | After a from-scratch re-embed, the OCSVM scores features from a different coordinate frame | critical |
| 3 | Post-alarm refills skip most of the stream | critical |
| 4 | The reference outlier rate is measured in-sample | high |
| 5 | The Z-test's alpha does not control the stream's false-alarm rate | high |
| 6 | The first reference is much larger than all later ones | medium |
| 7 | Right after a retrain the test is blind | medium |
| 8 | 2 epochs / 32 dims: the embedding may not have learned the graph | high |
| 9 | Only triples with new entities update the embedding | medium |
| 10 | The KG state used for training is not the real KG at that time | medium |
| 11 | Unscaled Hadamard features make the RBF kernel nearly constant | high |
| 12 | Latent bug: year parsed from an epoch timestamp | low |

In [ ]:
import os, sys, math, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'phase1' / 'drift_sphere.py').exists())
sys.path.insert(0, str(REPO / 'phase1'))
import drift_sphere as ds   # Phase 1 port of Antoine's notebook (same functions and loop)
import configs
import analyze

DATA = REPO / 'data'
RUNS = Path(os.environ.get('AUDIT_RUNS', REPO / 'phase1' / 'runs'))
QUALITY = Path(os.environ.get('AUDIT_QUALITY', REPO / 'phase1' / 'quality'))
RUN_HEAVY = False
rng = np.random.default_rng(0)

def finished_runs():
    if not RUNS.exists():
        return []
    return sorted(d for d in RUNS.iterdir() if (d / 'summary.json').exists())

def summaries():
    rows = [json.loads((d / 'summary.json').read_text()) for d in finished_runs()]
    df = pd.DataFrame(rows)
    if not df.empty:
        df['policy'] = df['run'].str.split('__').str[3]
    return df

S = summaries()
print('repository:', REPO)
print('finished Phase 1 runs found:', len(S))

In [ ]:
# Data shared by the demonstrations: CASE1 (11 disease types, cutoff 2002, switch 2005), as in Phase 1.
case1 = configs.DATASETS['case1']
triples_c1 = ds.load_temporal_triples(DATA / case1['file'])
init_kg_c1, init_stream_c1, future_c1, lookup_c1 = ds.build_initial_knowledge_graph(
    triples_c1, case1['cutoff_year'], case1['target_type'])
kg_c1 = ds.KnowledgeGraph(init_kg_c1)
print(f'CASE1: initial KG {len(init_kg_c1):,} triples, initial stream {len(init_stream_c1):,}, future stream {len(future_c1):,}')

def random_embedding(kg, seed, dim=64):
    """Stand-in for a from-scratch re-embedding: unrelated unit vectors (what a new frame looks like)."""
    return ds._random_embeddings(kg, {'seed': seed, 'real_dim': dim})

def features(rows, emb, composition):
    return ds.rows_to_features(rows, emb, composition)

PHASE1_OCSVM = dict(nu=0.1, gamma='scale')
def fit_ocsvm(X, standardize=True, gamma='scale'):
    scaler = StandardScaler().fit(X) if standardize else None
    Xs = scaler.transform(X) if scaler is not None else X
    model = OneClassSVM(kernel='rbf', nu=0.1, gamma=gamma).fit(Xs)
    return model, scaler

def outlier_rate(model, scaler, X):
    Xs = scaler.transform(X) if scaler is not None else X
    return float(np.mean(model.predict(Xs) == -1))

def adazor_threshold(n, alpha=0.025):
    return stats.norm.ppf(1 - alpha / 2) / math.sqrt(2 * n)

## 1. The sphere retrain rule cannot fire

**Claim.** The rule retrains the OCSVM when `distance_to_center / radius > 1`, where the center is the middle of the axis-aligned bounding box of **all** KG triple features and the radius is **half the box diagonal**. A ball with that radius contains the whole box, so every point inside the box has ratio ≤ 1. A new triple would have to fall outside the box *and* beyond the corners' distance. With a box built from hundreds of thousands of triples of every relation type, this practically never happens.

**Antoine's code** (reference notebook, cell 4):
```python
def boundary_from_min_max(min_point, max_point):
    center = 0.5 * (min_point + max_point)
    diagonal_length = float(np.linalg.norm(max_point - min_point))
    radius = float(diagonal_length / 2.0)
...
'threshold_exceeded_center_radius': bool(np.isfinite(center_radius_ratio) and center_radius_ratio > 1.0),
```
and in `process_triple_for_features`, the new triple (under the **new** embedding) is compared with `pre_unseen_boundary` (built under the **old** embedding).

**Proof.** For x in the box, each coordinate satisfies |x_i − c_i| ≤ (M_i − m_i)/2, so ‖x − c‖² ≤ Σ((M_i − m_i)/2)² = r². Hence ratio ≤ 1.

In [ ]:
# (a) Points inside a box: the ratio is at most 1, and in 64 dimensions typically far below.
d = 64
lo, hi = -rng.random(d), rng.random(d)
box = ds.boundary_from_min_max(lo, hi)
pts = lo + (hi - lo) * rng.random((100_000, d))
ratios_uniform = np.linalg.norm(pts - box['center'], axis=1) / box['radius']
print(f'uniform points in a {d}-d box: max ratio {ratios_uniform.max():.3f}, median {np.median(ratios_uniform):.3f}')

# (b) Real CASE1 triples: box from all KG features under embedding A, new stream triples under a fresh embedding B
for comp in ['hadamard', 'concatenation']:
    emb_A = random_embedding(kg_c1, seed=1)
    box_A = ds.compute_feature_space_boundary_from_kg(kg_c1, emb_A, comp)
    kg_next = ds.KnowledgeGraph(init_kg_c1)
    rows = future_c1.head(300).to_dict('records')
    for r in rows:
        kg_next.add(r)
    emb_B = random_embedding(kg_next, seed=2)
    Xb = features(rows, emb_B, comp)
    r_ = np.linalg.norm(Xb - box_A['center'], axis=1) / box_A['radius']
    print(f'{comp:14s} new triples under a new embedding vs old box: max ratio {r_.max():.3f}, median {np.median(r_):.3f}, share > 1: {np.mean(r_ > 1):.3f}')

In [ ]:
# Phase 1 evidence: the ratio logged at every new-entity re-embed, all models and compositions.
shift_files = [d / 'embedding_shift_metrics.tsv' for d in finished_runs()]
shift = pd.concat([pd.read_csv(f, sep='\t').assign(run=f.parent.name) for f in shift_files if f.exists() and f.stat().st_size > 1],
                  ignore_index=True) if shift_files else pd.DataFrame()
if shift.empty:
    print('Phase 1 runs not available yet.')
else:
    u = shift[shift['trigger_reason'].eq('unseen_element')]
    u = u.assign(model=u['run'].str.split('__').str[1], composition=u['run'].str.split('__').str[2])
    print(f'{len(u):,} new-entity re-embeds logged; share with ratio > 1: {(u["center_radius_ratio"] > 1).mean():.4f}')
    display(u.groupby(['model', 'composition'])['center_radius_ratio'].agg(['count', 'median', 'max']).round(3))
    u['center_radius_ratio'].hist(bins=50); plt.axvline(1, color='red'); plt.xlabel('distance to center / radius'); plt.title('Sphere ratio at every re-embed (Phase 1)'); plt.show()

**Consequence.** At k = 1 the rule is equivalent to "never retrain the OCSVM after a new-entity re-embed". **Fix.** Replace the geometric test with a test of what matters, whether the old detector's verdicts change on unchanged triples (Phase 2, churn rule), or keep one coordinate frame so the question does not arise.

## 2. After a re-embed, the OCSVM scores features from a different coordinate frame

**Claim.** Every triple that brings a new entity re-trains the whole embedding from scratch. Independently trained embeddings are only defined up to rotations, reflections and other symmetries of the scoring function, and training randomness makes them differ further. The OCSVM, trained on features from the old embedding, then scores features from the new one. Since the sphere rule never fires (problem 1), this happens at every re-embed, and the outlier rate jumps although the data did not change.

**Antoine's code** (cell 5): on an unseen element, `embedding_state = generate_transe_embeddings_from_kg(kg_df, ...)` is called, and the next window is featurised with the new state, while `ocsvm` is only retrained if `threshold_exceeded_center_radius`.

**Demonstration.** Same triples, same OCSVM; only the coordinate frame changes. We even use the mildest possible change: the *same* embedding, rotated by a random orthogonal matrix (every distance between entities is preserved).

In [ ]:
def rotate(emb, seed):
    Q, _ = np.linalg.qr(np.random.default_rng(seed).normal(size=(emb.dim, emb.dim)))
    return ds.EmbeddingState(emb.entity_index, emb.entity_matrix @ Q, emb.relation_index, emb.relation_matrix @ Q,
                             emb.dim, emb.kg_rows, 0.0)

ref_rows = init_stream_c1.tail(250).to_dict('records')            # reference window (pre-switch)
new_rows = future_c1.head(250).to_dict('records')                 # next 250 triples (still before the 2005 switch)
kg_all = ds.KnowledgeGraph(init_kg_c1)
for r in new_rows:
    kg_all.add(r)
emb = random_embedding(kg_all, seed=3)
rows_out = []
for comp in ['hadamard', 'concatenation']:
    model, scaler = fit_ocsvm(features(ref_rows, emb, comp))
    base = outlier_rate(model, scaler, features(new_rows, emb, comp))
    rot = outlier_rate(model, scaler, features(new_rows, rotate(emb, 7), comp))
    new = outlier_rate(model, scaler, features(new_rows, random_embedding(kg_all, seed=11), comp))
    p_ref = outlier_rate(model, scaler, features(ref_rows, emb, comp))
    thr = adazor_threshold(250)
    rows_out.append({'composition': comp, 'reference (in-sample)': p_ref, 'same frame': base,
                     'rotated frame': rot, 'new frame': new, 'ADAZOR threshold': thr,
                     'alarm, same frame': abs(base - p_ref) > thr, 'alarm, rotated': abs(rot - p_ref) > thr,
                     'alarm, new frame': abs(new - p_ref) > thr})
pd.DataFrame(rows_out).round(3)

In [ ]:
# Phase 1 evidence: alarms vs re-embeds, Antoine's rule (sphere1) vs always retrain.
if S.empty:
    print('Phase 1 runs not available yet.')
else:
    cols = ['dataset', 'model', 'composition', 'policy', 'num_detected_drifts', 'num_unseen_refresh_events',
            'num_threshold_retrains', 'tested_steps', 'future_rows']
    t = S[S['policy'].isin(['sphere1', 'always'])][cols].sort_values(['dataset', 'policy', 'model', 'composition'])
    display(t)
    share = []
    for d in finished_runs():
        summ, cfg, hist, sh = analyze.load_run(d)
        m = analyze.alarm_metrics(hist, sh, None, cfg['window_size'])
        share.append({'run': d.name, 'policy': d.name.split('__')[3], 'alarms_after_reembed_share': m.get('alarms_after_reembed_share')})
    display(pd.DataFrame(share).groupby('policy')['alarms_after_reembed_share'].describe()[['count', 'mean', 'min', 'max']].round(3))

**Consequence.** With Antoine's rule, alarms track re-embeds rather than drift; in Phase 1 the counts are identical across embedding models and compositions (and the same as with random vectors). **Fix.** Keep a single coordinate frame while monitoring (fixed embedding + inductive placement of new entities, or warm-start / alignment), and re-validate the detector after any deliberate re-embed.

## 3. Post-alarm refills skip most of the stream

**Claim.** After every alarm, 25 triples are kept and **225 are pulled from the stream without being tested** (and without re-embedding). With an alarm at almost every re-embed (problem 2), most of the stream is never tested, possibly including the drift switch itself.

**Antoine's code** (cell 5):
```python
keep_count = max(1, int(round(window_size * keep_fraction_after_drift)))
...
while len(rebuilt_raw_rows) < window_size and i < len(future_records):
    next_row = pd.Series(future_records[i]); i += 1
    kg_df, lookup = update_knowledge_graph(kg_df, pd.DataFrame([next_row]), lookup)
    rebuilt_raw_rows.append(next_row.to_dict())
```

In [ ]:
if S.empty:
    print('Phase 1 runs not available yet.')
else:
    t = S[['dataset', 'model', 'composition', 'policy', 'num_detected_drifts', 'tested_steps', 'future_rows']].copy()
    t['share_tested'] = (t['tested_steps'] / t['future_rows']).round(3)
    # Was the switch itself tested? Nearest tested triple to the switch, per run.
    near = []
    for d in finished_runs():
        summ, cfg, hist, sh = analyze.load_run(d)
        if cfg.get('switch_year') is None or hist.empty:
            continue
        stream = {'case1': future_c1}.get(cfg['dataset'])
        if stream is None:
            continue
        switch_pos = int(np.argmax(stream['event_year'].to_numpy() >= cfg['switch_year'])) + 1
        tested = hist['stream_index'].to_numpy()
        after = tested[tested >= switch_pos]
        near.append({'run': d.name, 'switch_stream_index': switch_pos,
                     'first_tested_after_switch': int(after[0]) if len(after) else None,
                     'gap': int(after[0] - switch_pos) if len(after) else None})
    display(t.sort_values(['dataset', 'policy', 'model']).head(40))
    if near:
        display(pd.DataFrame(near))

**Consequence.** Detection delay and peak statistics are measured on a small, irregular subsample of the stream. **Fix.** Keep testing during refills (or refill from a separate buffer), and report the share of the stream actually tested.

## 4. The reference outlier rate is measured in-sample

**Claim.** The reference rate is the OCSVM's outlier rate **on its own training points**, which is ≈ ν by construction. Every current window is scored **out-of-sample**. Even with no drift and no re-embedding, out-of-sample points are flagged more often, so the test compares two different quantities.

**Antoine's code** (cell 4):
```python
ocsvm.fit(X_ref)
flags = outlier_flags_from_ocsvm(ocsvm, X_ref)     # in-sample
...
detector.reset(new_reference_flags)
```

**Demonstration.** No drift by construction: reference and current windows are both random draws from the **same** pre-switch CASE1 triples, under **one fixed** embedding. We repeat 200 times.

In [ ]:
pool = init_stream_c1.to_dict('records')            # all pre-cutoff disease triples (one period, no drift)
emb_fixed = random_embedding(kg_c1, seed=5)
res = []
for comp in ['hadamard', 'concatenation']:
    Xpool = features(pool, emb_fixed, comp)
    for rep in range(200):
        idx = rng.permutation(len(Xpool))
        ref, cur = Xpool[idx[:250]], Xpool[idx[250:500]]
        model, scaler = fit_ocsvm(ref)
        p_in, p_out = outlier_rate(model, scaler, ref), outlier_rate(model, scaler, cur)
        res.append({'composition': comp, 'in_sample': p_in, 'out_of_sample': p_out,
                    'alarm': abs(p_out - p_in) > adazor_threshold(250)})
res = pd.DataFrame(res)
print(f'pool size: {len(pool)} triples (windows of 250 drawn without replacement)')
res.groupby('composition').agg(in_sample=('in_sample', 'mean'), out_of_sample=('out_of_sample', 'mean'),
                               false_alarm_share=('alarm', 'mean')).round(3)

In [ ]:
# Phase 1 evidence: mean reference vs mean current outlier rate (always-retrain runs have no frame mismatch).
if S.empty:
    print('Phase 1 runs not available yet.')
else:
    rows = []
    for d in finished_runs():
        h = pd.read_csv(d / 'window_history.tsv.gz', sep='\t')
        rows.append({'run': d.name, 'policy': d.name.split('__')[3], 'mean_reference_rate': h['reference_outlier_fraction'].mean(),
                     'mean_current_rate': h['current_outlier_fraction'].mean()})
    display(pd.DataFrame(rows).groupby('policy')[['mean_reference_rate', 'mean_current_rate']].mean().round(3))

**Consequence.** False alarms even when nothing changes. **Fix.** Estimate the reference rate out-of-sample: split the reference into a training part and a calibration part (conformal p-values, Bates et al. 2023), or cross-validate.

## 5. The Z-test's α does not control the stream's false-alarm rate

**Claim.** ADAZOR's threshold is `z_{1−α/2} / sqrt(2n)`. This is the two-proportion test with the variance at its worst case p = 0.5 and with the reference size ignored. At the actual outlier rate (≈ 0.1) it is very conservative per test. But the test is run **at every new triple on windows that share 249 of 250 triples**, so per-test α says nothing about how often a stream raises a false alarm. What actually drives alarms is any systematic offset (problems 2 and 4), not noise.

**Antoine's code** (fallback detector, cell 1, same formula as `adazor.detectors.ZTestDriftDetector`):
```python
threshold = st.norm().ppf(1 - self.alpha / 2) / math.sqrt(2 * n)
return bool(abs(p_outliers - p_orig) > threshold)
```

In [ ]:
n, alpha = 250, 0.025
thr = adazor_threshold(n, alpha)
p = 0.10
pooled = stats.norm.ppf(1 - alpha / 2) * math.sqrt(p * (1 - p) * 2 / n)
per_test = stats.binom.sf(math.floor((p + thr) * n), n, p)          # P(current rate > p_ref + thr) if both are truly 0.1
print(f'ADAZOR threshold at n={n}: {thr:.4f}  |  pooled-variance threshold at p=0.1: {pooled:.4f}')
print(f'per-test false-alarm probability at p=0.1: {per_test:.2e} (nominal alpha = {alpha})')

def stream_alarm_prob(true_rate, ref_rate=0.10, length=2000, reps=300):
    hits = 0
    for _ in range(reps):
        x = rng.random(length + n) < true_rate
        csum = np.concatenate([[0], np.cumsum(x)])
        win = (csum[n:] - csum[:-n]) / n                     # every-step sliding window
        hits += np.any(np.abs(win - ref_rate) > thr)
    return hits / reps

print('probability of at least one alarm over 2,000 tested triples, reference rate 0.10:')
for true_rate in [0.10, 0.15, 0.20, 0.25]:
    print(f'  true out-of-sample rate {true_rate:.2f}: {stream_alarm_prob(true_rate):.2f}')

**Consequence.** α = 0.025 is not the false-alarm rate. If the real outlier rate is 0.1 the test essentially never fires by chance; if a systematic bias moves it to 0.2–0.25 it fires with certainty. **Fix.** Remove the biases (problems 2, 4), then calibrate the threshold to a target average run length on no-drift streams, or use a sequential test designed for streams (Bernoulli CUSUM, conformal martingale).

## 6. The first reference is much larger than all later ones

**Claim.** The first OCSVM is trained on the whole initial stream except the last 250 triples; after the first alarm, the reference is 250 triples. Detector behaviour therefore differs before and after the first alarm.

**Antoine's code** (cell 5):
```python
reference_raw_df = initial_stream.iloc[:-seed_window_size]
```

In [ ]:
sizes = [{'dataset': 'case1 (disease types)', 'initial_stream': len(init_stream_c1), 'first_reference': len(init_stream_c1) - 250}]
g = configs.DATASETS['case1_gene']
_, s_gene, _, _ = ds.build_initial_knowledge_graph(triples_c1, g['cutoff_year'], g['target_type'])
sizes.append({'dataset': "case1, Antoine's Gene-Disease stream", 'initial_stream': len(s_gene), 'first_reference': len(s_gene) - 250})
if RUN_HEAVY:
    f = configs.DATASETS['full']
    tf = ds.load_temporal_triples(DATA / f['file'])
    _, s_full, _, _ = ds.build_initial_knowledge_graph(tf, f['cutoff_year'], f['target_type'])
    sizes.append({'dataset': 'full data, Gene-Disease', 'initial_stream': len(s_full), 'first_reference': len(s_full) - 250})
else:
    sizes.append({'dataset': 'full data, Gene-Disease (Phase 1 log / edition 1 output)', 'initial_stream': 23186, 'first_reference': 22936})
pd.DataFrame(sizes).assign(later_references=250)

**Fix.** Use a reference of fixed size from the start (e.g. the last 250 pre-cutoff triples).

## 7. Right after a retrain the test is blind

**Claim.** After an alarm, the current window is set to the new reference itself, so the first tests compare the reference with (almost) itself. The test cannot fire until enough new triples have replaced the reference rows, and then fires mechanically once the window is out-of-sample (problem 4).

**Antoine's code** (cell 5):
```python
current_raw_window = deque(reference_raw_df.tail(min(window_size, len(reference_raw_df))).to_dict('records'), maxlen=window_size)
```

In [ ]:
cand = [d for d in finished_runs() if '__always__' in d.name] or finished_runs()
if not cand:
    print('Phase 1 runs not available yet.')
else:
    d = cand[0]
    h = pd.read_csv(d / 'window_history.tsv.gz', sep='\t')
    fig, ax = plt.subplots(figsize=(11, 3.5))
    ax.plot(h['stream_index'], h['current_outlier_fraction'], lw=1, label='current window')
    ax.plot(h['stream_index'], h['reference_outlier_fraction'], lw=1, label='reference (in-sample)')
    a = h[h['drift_detected']]
    ax.scatter(a['stream_index'], a['current_outlier_fraction'], color='red', s=12, label='alarm')
    ax.set_title(d.name, fontsize=9); ax.set_xlabel('stream triple'); ax.legend(fontsize=8); plt.show()

**Fix.** Keep the current window and the reference disjoint (test only on triples that arrived after the reference).

## 8. With 2 epochs and 32 dimensions, the embedding may not have learned the graph

**Claim.** The reference notebook trains each model for 2 epochs at dimension 32 (and does not move the model to the GPU). If the embedding has not learned the graph, the features mostly encode random initialisation, and differences between models are not about knowledge.

**Antoine's code** (cell 1): `EMBEDDING_DIM = 32`, `EMBEDDING_EPOCHS = 2`.

**Evidence.** Link-prediction MRR on held-out cutoff triples (`phase1/embedding_quality.py`), compared with random guessing (≈ ln(N)/N for N entities). A preliminary local check of DistMult at the notebook's settings on CASE1 gave MRR ≈ 0.0001, below random guessing (≈ 0.00016); the Phase 1 run at 10 epochs / 64 dims is shown below.

In [ ]:
qfiles = sorted(QUALITY.glob('*.json')) if QUALITY.exists() else []
if not qfiles:
    print('MRR results not available yet (phase1/quality/).')
else:
    q = pd.DataFrame([json.loads(f.read_text()) for f in qfiles])
    if 'random_mrr_approx' in q:
        q['mrr_vs_random'] = (q['mrr'] / q['random_mrr_approx']).round(1)
    cols = ['dataset', 'model', 'preset', 'epochs', 'embedding_dim', 'mrr', 'hits_at_10', 'random_mrr_approx', 'mrr_vs_random']
    display(q[[c for c in cols if c in q]])
print('To compare with the notebook settings on the server:  python phase1/embedding_quality.py --dataset case1 --preset antoine')

## 9. Only triples with new entities update the embedding

**Claim.** The embedding is re-trained only when a triple contains an unseen entity or relation. Triples between known entities, the majority, never change it, however many accumulate. When the embedding is updated is decided by vocabulary novelty, not by how much the graph changed.

In [ ]:
lk = {'nodes': set(lookup_c1['nodes']), 'edge_types': set(lookup_c1['edge_types'])}
flags = []
for r in future_c1.to_dict('records'):
    flags.append(ds.triple_has_unseen_elements(r, lk))
    lk['nodes'].update([r['subject'], r['object']]); lk['edge_types'].add(r['edge_type'])
flags = np.array(flags)
print(f'CASE1 stream: {flags.mean():.1%} of triples bring a new entity (trigger a re-embed); {1 - flags.mean():.1%} never update the embedding')
yr = future_c1['event_year'].astype(int)
pd.DataFrame({'year': yr, 'new_entity': flags}).groupby('year')['new_entity'].agg(['size', 'mean']).rename(
    columns={'size': 'triples', 'mean': 'share_with_new_entity'}).round(3).T

**Note.** The share of triples with new entities is itself a meaningful novelty signal, and a natural, embedding-free drift statistic (Phase 2).

## 10. The KG state used for training is not the real KG at that time

**Claim.** (a) All undated triples are placed in the initial KG, as if known from the start. (b) After the cutoff, only triples of the monitored type(s) are added; every other relation type stays frozen at the cutoff. The embedding is therefore trained on a graph that never existed.

**Antoine's code** (cell 2):
```python
initial_mask = triples_df['event_year'].isna() | (triples_df['event_year'] <= cutoff_year)
...
future_stream = future_df[future_df['type'].eq(target_type)]
```

In [ ]:
undated = triples_c1['event_year'].isna()
future_all = triples_c1[~undated & (triples_c1['event_year'] > case1['cutoff_year'])]
excluded = future_all[~future_all['type'].isin(case1['target_type'])]
print(f'CASE1: {undated.mean():.1%} of all triples are undated and placed in the initial KG')
print(f'after the cutoff: {len(future_all):,} dated triples, of which {len(excluded):,} ({len(excluded)/len(future_all):.1%}) never enter the KG (non-monitored types)')
excluded['type'].value_counts().head(8)

## 11. Unscaled Hadamard features make the RBF kernel nearly constant

**Claim.** Every entity and relation vector is normalised to length 1. A Hadamard feature s∘p∘o then has tiny coordinates (≈ d^(−3/2) each), so squared distances between triples are tiny and, with γ = 0.5 and no scaling (the notebook's settings), the RBF kernel is ≈ 1 for every pair: the OCSVM sees almost no structure. Concatenation behaves completely differently at the same γ, so comparing compositions (or models) partly compares feature scales.

**Antoine's code** (cell 1, cell 4): `GAMMA = 0.5`, `STANDARDIZE = False`; `return s * p * o`.

In [ ]:
emb32 = random_embedding(kg_c1, seed=9, dim=32)
rows = init_stream_c1.sample(500, random_state=0).to_dict('records')
out = []
for comp in ['hadamard', 'concatenation']:
    X = features(rows, emb32, comp)
    d2 = ((X[:, None, :] - X[None, :, :]) ** 2).sum(-1)[np.triu_indices(len(X), 1)]
    K = np.exp(-0.5 * d2)
    out.append({'composition': comp, 'mean |coordinate|': np.abs(X).mean(), 'median squared distance': np.median(d2),
                'RBF kernel (gamma=0.5) min': K.min(), 'median': np.median(K), 'max': K.max()})
pd.DataFrame(out).round(5)

**Fix.** Standardise features on the reference and set γ relative to the data (`gamma='scale'` or the median heuristic), as in the Phase 1 configuration.

## 12. Latent bug: year parsed from an epoch timestamp

**Claim.** When `year` is missing but `timestamp` is present, the year is taken as the first four digits of the timestamp string. Timestamps are Unix epoch seconds, so this gives nonsense years. It does not trigger in the current files (undated rows have no timestamp either), but it would silently mis-order any file where only the timestamp is filled.

**Antoine's code** (cell 2, unchanged in the Phase 1 port):
```python
timestamp_year = pd.to_numeric(df['timestamp'].astype(str).str.extract(r'(\d{4})', expand=False), errors='coerce')
return year.fillna(timestamp_year)
```

In [ ]:
toy = pd.DataFrame({'year': [1972.0, np.nan], 'timestamp': ['90115200', '90115200']})
print('parsed years:', ds.infer_event_year(toy).tolist(), '-> the second row should be 1972 (epoch 90115200 = 9 Nov 1972)')
print('rows in CASE1 where the fallback is used:', int((triples_c1['year'].isna() & triples_c1['timestamp'].ne('')).sum()))

## Summary

| # | Problem | Effect on Phase 1 results | Fix (Phase 2) |
|---|---|---|---|
| 1 | Sphere rule cannot fire | Antoine's rule = never retrain after a re-embed | Churn-based retrain test, or a single coordinate frame |
| 2 | New coordinate frame after every re-embed | Alarms track re-embeds, identical across models | Fixed embedding + inductive new entities; alignment; warm start |
| 3 | Refills skip the stream | Most triples, possibly the switch, never tested | Test during refill; report share tested |
| 4 | In-sample reference rate | False alarms with no drift | Held-out / conformal calibration |
| 5 | α ≠ stream false-alarm rate | Alarm frequency set by biases, not α | Calibrate to a target run length; sequential tests |
| 6 | Large first reference | Behaviour changes after the first alarm | Fixed-size reference |
| 7 | Blind after retrain | Dead zone, then mechanical alarms | Disjoint current and reference windows |
| 8 | Under-trained embedding | Features may be mostly random initialisation | Report MRR; train to convergence |
| 9 | Re-embed only on new entities | Embedding update tied to vocabulary novelty | Deliberate, scheduled updates |
| 10 | Unreal KG state | Training graph never existed | Add all dated triples; separate monitored stream |
| 11 | Unscaled Hadamard + fixed γ | Kernel nearly constant; compositions not comparable | Standardise; data-driven γ |
| 12 | Epoch-year parsing | None now (latent) | Parse epoch seconds as dates |

Problems 1–3 together explain the main Phase 1 observation: with Antoine's rule, the detector raises an alarm at almost every re-embed, independently of the embedding model, and most of the stream is consumed by refills.